In [1]:
#r "nuget: Microsoft.Data.Analysis, 0.22.2"
#r "nuget: Plotly.NET, 5.0.0"
#r "nuget: Plotly.NET.Interactive, 5.0.0"
#r "nuget: Plotly.NET.CSharp, 0.13.0"
#r "nuget: NUnit, 4.3.2"
#r "nuget: Combinatorics, 2.0.0"

Installed Packages Combinatorics, 2.0.0 Microsoft.Data.Analysis, 0.22.2 NUnit, 4.3.2 Plotly.NET, 5.0.0 Plotly.NET.CSharp, 0.13.0 Plotly.NET.Interactive, 5.0.0

Loading extensions from `C:\Users\xumin\.nuget\packages\plotly.net.interactive\5.0.0\lib\netstandard2.1\Plotly.NET.Interactive.dll`

Loading extensions from `C:\Users\xumin\.nuget\packages\microsoft.data.analysis\0.22.2\interactive-extensions\dotnet\Microsoft.Data.Analysis.Interactive.dll`

In [2]:
using System.IO;
using System.Collections.Frozen;
using System.Collections.Generic;
using System.Text.RegularExpressions;
using Microsoft.Data.Analysis;
using Plotly.NET.CSharp;
using NUnit.Framework;
using NUnit.Framework.Legacy;

In [3]:
using container_size_t = int;
using filename_t = string;
using platform_t = string;
using action_t = string;
using time_collection = System.Collections.Generic.Dictionary<string, double[][]>;  // <filename_t, double[][]>
using time_set = System.Collections.Generic.Dictionary<string, double[]>;           // <action_t, double[]>

In [4]:
static class Check {
    public const container_size_t repetitions = 10;
    public const RegexOptions common_re_options = RegexOptions.Compiled | RegexOptions.IgnoreCase | RegexOptions.CultureInvariant | RegexOptions.NonBacktracking;
    public struct log_property_set_t {
        public string platform;
        public List<string[]> sequence;
        public string for_file;
    }
    public class Base {
        public static readonly Regex re_float = new(@"\d+(\.\d+)?", common_re_options);
        public log_property_set_t log_property;
        public container_size_t expected_filtered_line_count_per_repetition { get; protected set; }
        public Base(log_property_set_t log_property) {
            this.log_property = log_property;
        }
        public void line_count(IReadOnlyList<string> filtered_line) {
            ClassicAssert.AreEqual(expected_filtered_line_count_per_repetition * repetitions, filtered_line.Count);
        }
    }
    public class Lab : Base {
        static readonly FrozenDictionary<string, string> plain_pattern = new Dictionary<string, string>() {
            { "load_base_images", @"loadBaseImages done in" },
            { "extract", @"extractor done in" },
            { "create", @" done in" } // at this moment we failed to get function name for image creation so console prints no function name here
        }.ToFrozenDictionary();
        static readonly FrozenDictionary<string, Regex> re_pattern = new Dictionary<string, Regex>() {
            { "load_base_images", new Regex(@$"loadBaseImages done in ({Base.re_float.ToString()})\s*ms", common_re_options) },
            { "extract", new Regex(@$"extractor done in ({Base.re_float.ToString()})\s*ms", common_re_options) },
            { "create", new Regex(@$" done in ({Base.re_float.ToString()}?)\s*ms", common_re_options) } // at this moment we failed to get function name for image creation so console prints no function name here
        }.ToFrozenDictionary();
        public Lab(log_property_set_t log_property) : base(log_property) {
            expected_filtered_line_count_per_repetition = 2 + log_property.sequence.Count + log_property.sequence.Sum(sub_sequence => sub_sequence.Length);
        }
        public time_collection sep_and_order(IReadOnlyList<string> filtered_line) {
            time_collection result = new time_collection() {
                { "extract", new double[log_property.sequence.Count][] },
                { "create", new double[log_property.sequence.Count][] }
            };
            foreach (var action in result.Keys) {
                for (var i = 0; i < log_property.sequence.Count; ++i) { result[action][i] = Enumerable.Repeat<double>(Double.NaN, repetitions).ToArray<double>(); }
            }
            for (var rpt = 0; rpt < repetitions; ++rpt) { // repetition
                var short_off = rpt * expected_filtered_line_count_per_repetition;
                // sep
                Assert.That<string>(filtered_line[short_off + 0], Does.Match(plain_pattern["load_base_images"]));
                Assert.That<string>(filtered_line[short_off + 1], Does.Match(plain_pattern["extract"]));
                // order
                var off = 2;
                for (var i = 0; i < log_property.sequence.Count; ++i) { // file level operations
                    var sub_sequence_seg = log_property.sequence[i];
                    off += 1; // skip loadBaseImages
                    for (var j = 0; j < sub_sequence_seg.Length; ++j) { // cell level operations
                        var m = re_pattern[sub_sequence_seg[j]].Match(filtered_line[short_off + off]);
                        Assert.That(m.Success, Is.True);
                        var duration = double.Parse(m.Groups[1].Value) / 1000.0; // ms to s
                        result[sub_sequence_seg[j]][i][rpt] = duration;
                        off += 1;
                    }
                }
            }
            return result;
        }
    }
    public class RStudio : Base {
        static readonly FrozenDictionary<string, string> plain_pattern = new Dictionary<string, string>() {
            { "parse", @"Execution duration of function parse :" },
            { "extract", @"Execution duration of function extract :" },
            { "create", @"Execution duration of function create :" }
        }.ToFrozenDictionary();
        static readonly string[] cell_level_actions = new[] { "extract", "create", };
        public RStudio(log_property_set_t log_property) : base(log_property) {
            expected_filtered_line_count_per_repetition = 2 * (3 + log_property.sequence.Sum(sub_sequence => sub_sequence.Length));
        }
        public time_collection sep_and_order(IReadOnlyList<string> filtered_line) {
            time_collection result = new time_collection() {
                { "parse", new double[1][] },
                { "extract", new double[log_property.sequence.Count][] },
                { "create", new double[log_property.sequence.Count][] },
            };
            result["parse"][0] = new double[repetitions];
            foreach (var action in cell_level_actions) {
                for (var i = 0; i < log_property.sequence.Count; ++i) { result[action][i] = Enumerable.Repeat<double>(Double.NaN, repetitions).ToArray<double>(); }
            }
            for (var rpt = 0; rpt < repetitions; ++rpt) { // repetition
                var short_off = rpt * expected_filtered_line_count_per_repetition;
                // sep
                Assert.That<string>(filtered_line[short_off + 0], Does.Match(plain_pattern["extract"]));
                Assert.That<string>(filtered_line[short_off + 2], Does.Match(plain_pattern["extract"]));
                Assert.That<string>(filtered_line[short_off + 4], Does.Match(plain_pattern["parse"]));
                // order
                result["parse"][0][rpt] = double.Parse(Base.re_float.Matches(filtered_line[short_off + 5])[2].Value);
                var off = 6;
                for (var i = 0; i < log_property.sequence.Count; ++i) { // file level operations
                    var sub_sequence_seg = log_property.sequence[i];
                    for (var j = 0; j < sub_sequence_seg.Length; ++j) { // cell level operations
                        Assert.That<string>(filtered_line[short_off + off], Does.Match(plain_pattern[sub_sequence_seg[j]]));
                        var duration = double.Parse(Base.re_float.Matches(filtered_line[short_off + off + 1])[2].Value);
                        result[sub_sequence_seg[j]][i][rpt] = duration;
                        off += 2;
                    }
                }
            }
            return result;
        }
    }
}

In [5]:
IReadOnlyDictionary<filename_t, Check.log_property_set_t> log_property; {
    var _log_property = new Dictionary<filename_t, Check.log_property_set_t>(); {
        DataFrame df = DataFrame.LoadCsv("log-prop.csv", dataTypes: Enumerable.Repeat(typeof(string), 5).ToArray());
        foreach (var row_seg in df.Rows) {
            if (string.IsNullOrEmpty(row_seg[df.Columns.IndexOf("prefix")].ToString()) == false) {
                _log_property.Add(
                    row_seg[df.Columns.IndexOf("prefix")].ToString(),
                    new Check.log_property_set_t {
                        platform = row_seg[df.Columns.IndexOf("platform")].ToString(),
                        sequence = Regex.Split(row_seg[df.Columns.IndexOf("sequence")].ToString(), @",\s*").Where(row => !string.IsNullOrEmpty(row)).Select(row => Regex.Split(row, @"\s+")).ToList(),
                        for_file = row_seg[df.Columns.IndexOf("file")].ToString(),
                    }
                );
            }
        }
    }
    log_property = _log_property.ToFrozenDictionary();
}

Regex expected_RStudio_console_log = new Regex(@"Execution duration of function (parse|extract|create) :", Check.common_re_options);
string log_path = "log";
FileInfo[] log_files = new DirectoryInfo(log_path).GetFiles("*.log");
IDictionary<filename_t, time_collection> time_collections = new OrderedDictionary<filename_t, time_collection>();
foreach (var log_file in log_files) {
    Console.WriteLine($"Checking {log_file.Name}");
    string[] line = File.ReadAllLines(log_file.FullName).Where(l => !string.IsNullOrWhiteSpace(l)).ToArray();
    var basename = log_file.Name.Substring(0, log_file.Name.Length - ".log".Length);
    var current_log_property = log_property[basename];
    switch (current_log_property.platform) {
    case "lab.original": case "lab.vreapi": {
        var lab_tests = new Check.Lab(current_log_property);
        lab_tests.line_count(line);
        time_collections.Add(basename, lab_tests.sep_and_order(line));
    }
    break;
    case "rstudio": {
        var rstudio_tests = new Check.RStudio(current_log_property);
        List<string> filtered_line = new List<string>();
        for (var i = 0; i < line.Length; ++i) {
            if (expected_RStudio_console_log.IsMatch(line[i])) {
                filtered_line.Add(line[i]);     // function name
                filtered_line.Add(line[i + 2]); // time
                i += 2;                         // skip next 2 lines
            }
        }
        rstudio_tests.line_count(filtered_line);
        time_collections.Add(basename, rstudio_tests.sep_and_order(filtered_line));
    }
    break;
    }
}
Console.WriteLine("✅Done");

Checking 20250705-154630.log
Checking 20250705-185345.log
Checking 20250705-191844.log
Checking 20250706-011853.log
Checking 20250707-171456.log
Checking 20250707-174435.log
Checking 20250707-181404.log
Checking 20250707-183013.1.log
Checking 20250707-185916.log
Checking 20250709-004817.1.log
Checking 20250709-012130.1.log
Checking 20250711-022045.log
Checking 20250711-175404.log
Checking 20250711-180809.log
Checking 20250711-192010.log
Checking 20250712-013145.log
Checking 20250712-195549.log
Checking 20250712-202739.log
Checking 20250713-142707.log
Checking 20250713-150127.log
Checking 20250713-184748.log
Checking 20250714-010848.log
Checking 20250714-013914.log
Checking 20250714-020410.log
Checking 20250714-125545.log
Checking 20250714-133658.log
Checking 20250714-144327.log
✅Done


In [6]:
// foreach (var file_level_collection in time_collections) {
//     Console.WriteLine($"({file_level_collection.Key}, {log_property[file_level_collection.Key].for_file}, {log_property[file_level_collection.Key].platform}):");
//     foreach (var action in file_level_collection.Value.Keys) {
//         Console.WriteLine($"  {action}:");
//         for (var nth_cell = 0; nth_cell < file_level_collection.Value[action].Length; ++nth_cell) {
//             Console.WriteLine(string.Format("    {0} {1,1}: ", action == "parse" ? "file" : "cell", action == "parse" ? " " : nth_cell) + string.Join("", file_level_collection.Value[action][nth_cell].Select(x => string.Format("{0,10:N3}", x))));
//         }
//     }
// }

In [7]:
using percent_of_trunc_t = container_size_t;
using count_of_trunc_t = container_size_t;
using no_t = container_size_t;

In [8]:
var max_count_of_truncation = 2;
var pk_column_names = new string[] {
    "Notebook",
    "Cell No.",
    "Platform",
    "Action",
};
var pk_column_types = new Type[] {
    typeof(string),
    typeof(container_size_t),
    typeof(string),
    typeof(string),
};
var stat_column_names = new List<string>(); {
    for (var i = 0; i <= max_count_of_truncation; ++i) {
        stat_column_names.Add(string.Format("Min{0}", i > 0 ? $" ({10 * i}% Trunc)" : ""));
        stat_column_names.Add(string.Format("Max{0}", i > 0 ? $" ({10 * i}% Trunc)" : ""));
        stat_column_names.Add(string.Format("Ave{0}", i > 0 ? $" ({10 * i}% Trunc)" : ""));
    }
    stat_column_names.Add("Med");
    for (var i = 0; i <= max_count_of_truncation; ++i) {
        stat_column_names.Add(string.Format("Max Abs Dev{0}", i > 0 ? $" ({10 * i}% Trunc)" : ""));
        stat_column_names.Add(string.Format("Max Abs Dev from Med{0}", i > 0 ? $" ({10 * i}% Trunc)" : ""));
        stat_column_names.Add(string.Format("Max Rel Dev{0} %", i > 0 ? $" ({10 * i}% Trunc)" : ""));
        stat_column_names.Add(string.Format("Max Rel Dev from Med{0} %", i > 0 ? $" ({10 * i}% Trunc)" : ""));
    }
    for (var i = 0; i < Check.repetitions; ++i) { stat_column_names.Add(string.Format("t{0}", i)); }
}
var row_count = 0;
foreach (var pair in log_property) {
    switch (pair.Value.platform) {
    case "lab.original": case "lab.vreapi": {
        row_count += pair.Value.sequence.Select(sub_sequence => sub_sequence.Length).Sum();
        break;
    }
    case "rstudio": {
        row_count += pair.Value.sequence.Select(sub_sequence => sub_sequence.Length).Sum() + 1; // +1 for parsing the entire notebook
        break;
    }
    }
}
Console.WriteLine($"Row count: {row_count}");
DataFrame overall = new DataFrame();
foreach (var pair in pk_column_names.Zip(pk_column_types)) {
    if (pair.Second == typeof(string)) { overall.Columns.Add(new StringDataFrameColumn(pair.First, row_count)); } 
    else if (pair.Second == typeof(container_size_t)) { overall.Columns.Add(new PrimitiveDataFrameColumn<container_size_t>(pair.First, row_count)); }
}
foreach (var name in stat_column_names) { overall.Columns.Add(new PrimitiveDataFrameColumn<double>(name, row_count)); }

Row count: 294


In [9]:
using System.Numerics;

In [10]:
public class Stat<_Ty, _Fallty> where _Ty : IComparable<_Ty>, INumber<_Ty> where _Fallty : INumber<_Fallty> {
    public container_size_t n { get; }
    public _Ty min => trunc_min[0];
    public IReadOnlyList<_Ty> trunc_min { get; }
    public _Ty max => trunc_max[0];
    public IReadOnlyList<_Ty> trunc_max { get; }
    public _Fallty ave => trunc_ave[0];
    public IReadOnlyList<_Fallty> trunc_ave { get; }
    public _Fallty med { get; }
    public IReadOnlyList<_Fallty> max_abs_dev { get; }
    public IReadOnlyList<_Fallty> max_rel_dev { get; }
    public IReadOnlyList<_Fallty> max_abs_dev_from_med { get; }
    public IReadOnlyList<_Fallty> max_rel_dev_from_med { get; }
    public Stat(IList<_Ty> array, container_size_t max_count_of_truncation = 2) {
        if (array == null || array.Count < 1) throw new ArgumentException("Array must have at least 1 element.");
        if (max_count_of_truncation < 0) { max_count_of_truncation = 0; }
        if (max_count_of_truncation >= array.Count / 2) { max_count_of_truncation = array.Count / 2 - (array.Count % 2 == 1 ? 0 : 1); }
        var a = array.OrderBy(x => x).ToArray();
        n = a.Length;
        med = (array.Count % 2 == 1) ? _Fallty.CreateChecked(a[n / 2]) : (_Fallty.CreateChecked(a[n / 2 - 1]) + _Fallty.CreateChecked(a[n / 2])) / _Fallty.CreateChecked(2);
        var _trunc_ave = new _Fallty[max_count_of_truncation + 1];
        var _trunc_min = new _Ty[max_count_of_truncation + 1];
        var _trunc_max = new _Ty[max_count_of_truncation + 1];
        var _max_abs_dev = new _Fallty[max_count_of_truncation + 1];
        var _max_rel_dev = new _Fallty[max_count_of_truncation + 1];
        var _max_abs_dev_from_med = new _Fallty[max_count_of_truncation + 1];
        var _max_rel_dev_from_med = new _Fallty[max_count_of_truncation + 1];
        var S = new _Ty[n + 1];
        S[0] = _Ty.Zero;
        for (container_size_t i = 0; i < n; ++i) { S[i + 1] = S[i] + a[i]; }
        for (container_size_t k = 0; k <= max_count_of_truncation; ++k) {
            container_size_t left = k, right = n - k;
            _trunc_ave[k] = _Fallty.CreateChecked(S[right] - S[left]) / _Fallty.CreateChecked(right - left);
            _trunc_min[k] = a[left];
            _trunc_max[k] = a[right - 1];
            _max_abs_dev[k] = _Fallty.Max(_Fallty.Abs(_Fallty.CreateChecked(_trunc_min[k]) - _trunc_ave[k]), _Fallty.Abs(_Fallty.CreateChecked(_trunc_max[k]) - _trunc_ave[k]));
            _max_rel_dev[k] = _Fallty.Abs(_max_abs_dev[k] / _trunc_ave[k]);
            _max_abs_dev_from_med[k] = _Fallty.Max(_Fallty.Abs(_Fallty.CreateChecked(_trunc_min[k]) - med), _Fallty.Abs(_Fallty.CreateChecked(_trunc_max[k]) - med));
            _max_rel_dev_from_med[k] = _Fallty.Abs(_max_abs_dev_from_med[k] / med);
        }
        trunc_min = _trunc_min;
        trunc_max = _trunc_max;
        trunc_ave = _trunc_ave;
        max_abs_dev = _max_abs_dev;
        max_rel_dev = _max_rel_dev;
        max_abs_dev_from_med = _max_abs_dev_from_med;
        max_rel_dev_from_med = _max_rel_dev_from_med;
    }
}

In [11]:
bool contains_NaN<T>(IReadOnlyList<T> array) where T : IFloatingPoint<T> {
    for (var i = 0; i < array.Count; ++i) { if (T.IsNaN(array[i])) { return true; } }
    return false;
}
var current_row = 0;
foreach (var file_level_collection in time_collections) {
    foreach (var action in file_level_collection.Value.Keys) {
        for (var nth_cell = 0; nth_cell < file_level_collection.Value[action].Length; ++nth_cell) {
            Console.WriteLine($"({file_level_collection.Key}, {log_property[file_level_collection.Key].for_file}, {log_property[file_level_collection.Key].platform}): {action} {nth_cell}");
            overall.Columns["Notebook"][current_row] = log_property[file_level_collection.Key].for_file;
            overall.Columns["Cell No."][current_row] = nth_cell;
            overall.Columns["Platform"][current_row] = log_property[file_level_collection.Key].platform;
            overall.Columns["Action"][current_row] = action;
            var duration = file_level_collection.Value[action][nth_cell];
            var stat = new Stat<double, double>(duration, max_count_of_truncation);
            if (contains_NaN<double>(duration) == false) {
                for (container_size_t i = 0; i <= max_count_of_truncation; ++i) {
                    overall.Columns[string.Format("Min{0}", i > 0 ? $" ({10 * i}% Trunc)" : "")][current_row] = stat.trunc_min[i];
                    overall.Columns[string.Format("Max{0}", i > 0 ? $" ({10 * i}% Trunc)" : "")][current_row] = stat.trunc_max[i];
                    overall.Columns[string.Format("Ave{0}", i > 0 ? $" ({10 * i}% Trunc)" : "")][current_row] = stat.trunc_ave[i];
                    overall.Columns["Med"][current_row] = stat.med;
                    overall.Columns[string.Format("Max Abs Dev{0}", i > 0 ? $" ({10 * i}% Trunc)" : "")][current_row] = stat.max_abs_dev[i];
                    overall.Columns[string.Format("Max Abs Dev from Med{0}", i > 0 ? $" ({10 * i}% Trunc)" : "")][current_row] = stat.max_abs_dev_from_med[i];
                    overall.Columns[string.Format("Max Rel Dev{0} %", i > 0 ? $" ({10 * i}% Trunc)" : "")][current_row] = stat.max_rel_dev[i] * 100.0;
                    overall.Columns[string.Format("Max Rel Dev from Med{0} %", i > 0 ? $" ({10 * i}% Trunc)" : "")][current_row] = stat.max_rel_dev_from_med[i] * 100.0;
                }
                for (container_size_t i = 0; i < Check.repetitions; ++i) { overall.Columns[string.Format("t{0}", i)][current_row] = duration[i]; }
                ++current_row;
            }
        }
    }
}

(20250705-154630, D1, lab.original): extract 0
(20250705-154630, D1, lab.original): extract 1
(20250705-154630, D1, lab.original): extract 2
(20250705-154630, D1, lab.original): extract 3
(20250705-154630, D1, lab.original): create 0
(20250705-154630, D1, lab.original): create 1
(20250705-154630, D1, lab.original): create 2
(20250705-154630, D1, lab.original): create 3
(20250705-185345, D1, lab.vreapi): extract 0
(20250705-185345, D1, lab.vreapi): extract 1
(20250705-185345, D1, lab.vreapi): extract 2
(20250705-185345, D1, lab.vreapi): extract 3
(20250705-185345, D1, lab.vreapi): create 0
(20250705-185345, D1, lab.vreapi): create 1
(20250705-185345, D1, lab.vreapi): create 2
(20250705-185345, D1, lab.vreapi): create 3
(20250705-191844, D1, rstudio): parse 0
(20250705-191844, D1, rstudio): extract 0
(20250705-191844, D1, rstudio): extract 1
(20250705-191844, D1, rstudio): extract 2
(20250705-191844, D1, rstudio): extract 3
(20250705-191844, D1, rstudio): create 0
(20250705-191844, D1, r

In [12]:
Console.WriteLine(overall);
System.IO.Directory.CreateDirectory("export/time");
DataFrame.SaveCsv(overall, "export/time/overall.csv");

Notebook                           Cell No.                           Platform                           Action                             Min                                Max                                Ave                                Min (10% Trunc)                    Max (10% Trunc)                    Ave (10% Trunc)                    Min (20% Trunc)                    Max (20% Trunc)                    Ave (20% Trunc)                    Med                                Max Abs Dev                        Max Abs Dev from Med               Max Rel Dev %                      Max Rel Dev from Med %             Max Abs Dev (10% Trunc)            Max Abs Dev from Med (10% Trunc)   Max Rel Dev (10% Trunc) %          Max Rel Dev from Med (10% Trunc) % Max Abs Dev (20% Trunc)            Max Abs Dev from Med (20% Trunc)   Max Rel Dev (20% Trunc) %          Max Rel Dev from Med (20% Trunc) % t0                                 t1                                 t2                  

In [13]:
// var counts_of_truncation = Enumerable.Range(0, max_count_of_truncation + 1); // truncate highest and lowest 0, 1, 2, ... from statistics
var counts_of_truncation = new[] { 0, 2, }; // truncate highest and lowest 0, 1, 2, ... from statistics
var column_names_for_compact_overall = new List<string>(); {
    column_names_for_compact_overall.AddRange(pk_column_names);
    foreach (var i in counts_of_truncation) {
        column_names_for_compact_overall.Add(string.Format("Max{0}", i > 0 ? $" ({10 * i}% Trunc)" : ""));
    }
    column_names_for_compact_overall.Add("Med");
    foreach (var i in counts_of_truncation) {
        // column_names_for_compact_overall.Add(string.Format("Max Abs Dev from Med{0}", i > 0 ? $" ({10 * i}% Trunc)" : ""));
        column_names_for_compact_overall.Add(string.Format("Max Rel Dev from Med{0} %", i > 0 ? $" ({10 * i}% Trunc)" : ""));
    }
}
DataFrame compact_overall = new DataFrame(column_names_for_compact_overall.Select(name => overall.Columns[name]));
foreach (var column_name in column_names_for_compact_overall.Slice(4, column_names_for_compact_overall.Count - 4)) {
    for (var i = 0; i < compact_overall.Columns[column_name].Length; ++i) {
        compact_overall.Columns[column_name][i] = Math.Round((double)compact_overall.Columns[column_name][i], 3);
    }
}
Console.WriteLine(compact_overall);
DataFrame.SaveCsv(compact_overall, "export/time/compact_overall.csv");

Notebook                           Cell No.                           Platform                           Action                             Max                                Max (20% Trunc)                    Med                                Max Rel Dev from Med %             Max Rel Dev from Med (20% Trunc) % 
D1                                 0                                  lab.original                       extract                            1.112                              1.039                              0.986                              12.825                             9                                  
D1                                 1                                  lab.original                       extract                            1.186                              1.049                              0.994                              19.268                             5.652                              
D1                                 2                

In [14]:
var platforms = new platform_t[] { "lab.original", "lab.vreapi", "rstudio", };
var actions = new action_t[] { "parse", "extract", "create", };
var export_part = new Dictionary<(count_of_trunc_t, platform_t, action_t), DataFrame>();
foreach (var t in from count_of_truncation in counts_of_truncation from platform in platforms from action in actions select (count_of_truncation, platform, action)) {
    if (t.platform != "rstudio" && t.action == "parse") { continue; } // we do not need to explicitly parse the notebook first using the current Jupyter implementation
    var col_names = new List<string>(pk_column_names) {
        string.Format("Max{0}", t.count_of_truncation > 0 ? $" ({10 * t.count_of_truncation}% Trunc)" : ""),
        "Med",
        // string.Format("Max Abs Dev from Med{0}", t.count_of_truncation > 0 ? $" ({10 * t.count_of_truncation}% Trunc)" : ""),
        string.Format("Max Rel Dev from Med{0} %", t.count_of_truncation > 0 ? $" ({10 * t.count_of_truncation}% Trunc)" : ""),
    };
    DataFrame part = new DataFrame(col_names.Select(name => compact_overall.Columns[name]));
    var platform_filter = (PrimitiveDataFrameColumn<bool>)compact_overall.Columns["Platform"].ElementwiseEquals(t.platform);
    var action_filter = (PrimitiveDataFrameColumn<bool>)compact_overall.Columns["Action"].ElementwiseEquals(t.action);
    var filter = (PrimitiveDataFrameColumn<bool>)(platform_filter & action_filter);
    part = part[filter];
    export_part[(t.count_of_truncation, t.platform, t.action)] = part;
    DataFrame.SaveCsv(part, string.Format("export/time/{0}.{1}.{2}.csv", $"{10 * t.count_of_truncation}% trunc", t.platform, t.action));
}

In [15]:
var tolerance_pc = 0.1 * 100;
Console.WriteLine($"Tolerance: {tolerance_pc}%");
foreach (var pair in export_part) {
    if (pair.Key.Item3 == "parse" && pair.Key.Item2 == "rstudio") { continue; } // very short durations, fluctuation doesn't matter
    DataFrame part = pair.Value;
    Console.WriteLine(string.Format("({0}, {1}, {2}):", $"{10 * pair.Key.Item1}% Trunc", pair.Key.Item2, pair.Key.Item3));
    var dev_col_name = string.Format("Max Rel Dev from Med{0} %", pair.Key.Item1 > 0 ? $" ({10 * pair.Key.Item1}% Trunc)" : "");
    var dev_col = (PrimitiveDataFrameColumn<double>)part[dev_col_name];
    var number_of_cases_below_tolerance = part.Filter(dev_col.ElementwiseLessThanOrEqual(tolerance_pc)).Rows.Count;
    var number_of_cases_above_tolerance = part.Rows.Count - number_of_cases_below_tolerance;
    var pass_rate = (double)number_of_cases_below_tolerance / dev_col.Length * 100.0;
    var max_rel_dev_from_med = dev_col.Max();
    Console.WriteLine(
        string.Format(
            "  {0} cases in total, {1} cases ({2:N1}%) below tolerance, {3} cases above tolerance, max: {4:N1}%",
            dev_col.Length, number_of_cases_below_tolerance, pass_rate, number_of_cases_above_tolerance, max_rel_dev_from_med
        )
    );
}

Tolerance: 10%
(0% Trunc, lab.original, extract):
  49 cases in total, 9 cases (18.4%) below tolerance, 40 cases above tolerance, max: 94.9%
(0% Trunc, lab.original, create):
  46 cases in total, 32 cases (69.6%) below tolerance, 14 cases above tolerance, max: 41.5%
(0% Trunc, lab.vreapi, extract):
  49 cases in total, 11 cases (22.4%) below tolerance, 38 cases above tolerance, max: 75.8%
(0% Trunc, lab.vreapi, create):
  46 cases in total, 28 cases (60.9%) below tolerance, 18 cases above tolerance, max: 189.1%
(0% Trunc, rstudio, extract):
  49 cases in total, 27 cases (55.1%) below tolerance, 22 cases above tolerance, max: 39.6%
(0% Trunc, rstudio, create):
  46 cases in total, 40 cases (87.0%) below tolerance, 6 cases above tolerance, max: 45.5%
(20% Trunc, lab.original, extract):
  49 cases in total, 44 cases (89.8%) below tolerance, 5 cases above tolerance, max: 12.4%
(20% Trunc, lab.original, create):
  46 cases in total, 46 cases (100.0%) below tolerance, 0 cases above tolerance

In [16]:
container_size_t count_of_trunc = 2;
IDictionary<action_t, DataFrame> cmp = new Dictionary<action_t, DataFrame>();
var cmp_pk_column_names = new string[] {
    "Notebook",
    "Cell No.",
    "Action",
};
foreach (var action in new action_t[] { "extract", "create", }) {
    DataFrame df = new DataFrame(cmp_pk_column_names.Select(name => export_part[(count_of_trunc, platforms[0], action)].Columns[name]));
    foreach (var platform in platforms) {
        var Med = export_part[(count_of_trunc, platform, action)].Columns["Med"];
        Med.SetName(string.Format("Med ({0})", platform));
        df.Columns.Add(Med);
        Console.WriteLine(string.Format(
            $"({platform}, {action}){Environment.NewLine}"
            +
            "Min of med: {0:N3}" + Environment.NewLine
            +
            "Med of med: {1:N3}" + Environment.NewLine
            +
            "Max of med: {2:N3}" + Environment.NewLine
            ,
            Med.Min(),
            Med.Median(),
            Med.Max()
        ));
    }
    cmp[action] = df;
    DataFrame.SaveCsv(df, string.Format("export/time/cmp.{0}.csv", action));
}

(lab.original, extract)
Min of med: 0.975
Med of med: 1.132
Max of med: 1.448

(lab.vreapi, extract)
Min of med: 1.153
Med of med: 1.373
Max of med: 1.648

(rstudio, extract)
Min of med: 1.170
Med of med: 1.265
Max of med: 1.446

(lab.original, create)
Min of med: 5.157
Med of med: 5.641
Max of med: 5.985

(lab.vreapi, create)
Min of med: 5.267
Med of med: 5.486
Max of med: 5.830

(rstudio, create)
Min of med: 5.232
Med of med: 5.358
Max of med: 5.726



In [17]:
// Console.WriteLine(cmp["extract"].Columns[4]);

In [18]:
using Combinatorics.Collections;

In [19]:
var combos = new Combinations<container_size_t>(Enumerable.Range(cmp_pk_column_names.Length, platforms.Length), 2);
foreach (var action in new action_t[] { "extract", "create", }) {
    DataFrame diff = new DataFrame();
    foreach (var combo in combos) {
        var __ref = (PrimitiveDataFrameColumn<double>)cmp[action].Columns[combo[0]];
        var __diff = (PrimitiveDataFrameColumn<double>)cmp[action].Columns[combo[1]];
        var _diff = (PrimitiveDataFrameColumn<double>)(__diff - __ref);
        var rate = (PrimitiveDataFrameColumn<double>)_diff / __ref * 100.0;
        var column_name = $"[{action}] {__diff.Name} v. {__ref.Name} by cells";
        _diff.SetName($"{column_name}");
        rate.SetName($"{column_name} (%)");
        diff.Columns.Add(_diff);
        diff.Columns.Add(rate);
        Console.WriteLine(column_name);
        // foreach (var r in rate) { Console.WriteLine(r); }
        Console.WriteLine(string.Format(
            "Med: {0:N3}" + Environment.NewLine
            +
            "Max: {1:N3}" + Environment.NewLine
            +
            "Med: {2:N1} %" + Environment.NewLine
            +
            "Max: {3:N1} %" + Environment.NewLine
            ,
            _diff.Median(),
            _diff.Max(),
            rate.Median(),
            rate.Max()
        ));
    }
    DataFrame.SaveCsv(diff, $"export/time/diff.{action}.csv");
    DataFrame result = cmp[action].Join(diff, "", "", JoinAlgorithm.Inner);
    DataFrame.SaveCsv(result, $"export/time/result.{action}.csv");
}

[extract] Med (lab.vreapi) v. Med (lab.original) by cells
Med: 0.204
Max: 0.559
Med: 17.9 %
Max: 54.4 %

[extract] Med (rstudio) v. Med (lab.original) by cells
Med: 0.137
Max: 0.314
Med: 12.0 %
Max: 31.8 %

[extract] Med (rstudio) v. Med (lab.vreapi) by cells
Med: -0.086
Max: 0.080
Med: -6.4 %
Max: 6.9 %

[create] Med (lab.vreapi) v. Med (lab.original) by cells
Med: -0.220
Max: 0.436
Med: -3.9 %
Max: 8.5 %

[create] Med (rstudio) v. Med (lab.original) by cells
Med: -0.266
Max: 0.298
Med: -4.8 %
Max: 5.7 %

[create] Med (rstudio) v. Med (lab.vreapi) by cells
Med: -0.066
Max: 0.207
Med: -1.2 %
Max: 3.9 %

